# Dispute Case Tracker
Gmail → SQLite → Google Sheets → Slack pipeline for `account.receivable.vn@grabtaxi.com`

**Auth:** OAuth 2.0 (Desktop app). First run opens a browser once to approve access; all subsequent runs are fully headless.

**Sections:**
1. Imports & Configuration
2. Queue Classification
3. Database
4. Gmail Auth & Helpers
5. Core Polling
6. Archive
7. Slack Notifications
7.5. Google Sheets Sync
8. Actions (Assign / Complete / Reclassify)
9. Run: Poll Once
10. Run: List Cases
11. Diagnostics
12. Tests

---
## 1. Imports & Configuration

In [49]:
#install requests package
import sys
!{sys.executable} -m pip install requests

In [50]:
#import requests and print its file path to confirm installation
import requests
print(requests.__file__)

c:\Users\vy.nguyenth\miniconda3\envs\vscode-env\Lib\site-packages\requests\__init__.py


In [51]:
#checking kernel executable and requests import to troubleshoot import error

import sys
print("Kernel executable:", sys.executable)
try:
    import requests
    print("requests path:", requests.__file__)
except Exception as e:
    print("Import error:", type(e).__name__, e)

Kernel executable: c:\Users\vy.nguyenth\miniconda3\envs\vscode-env\python.exe
requests path: c:\Users\vy.nguyenth\miniconda3\envs\vscode-env\Lib\site-packages\requests\__init__.py


In [52]:
#import libraries and configurations for Gmail API authentication and email processing

import os
import sys
import time
import sqlite3
import logging
import base64
import requests
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger(__name__)

print("Imports OK")

Imports OK


In [53]:
# ── Edit these values before running ──────────────────────────────────────────
CONFIG = {
    "GROUP_EMAIL": "account.receivable.vn@grabtaxi.com",
    "PROCESSED_LABEL": "dispute-tracker-processed",
    "POLL_INTERVAL_MINUTES": 5,
    "ARCHIVE_AFTER_DAYS": 1,
    "MAX_RESULTS": 100,
    "TIMEOUT_BUFFER_SECONDS": 5 * 60,
    "DB_FILE": os.path.join(os.getcwd(), "cases.db"),
    "CREDENTIALS_FILE": os.path.join(os.getcwd(), "credentials.json"),
    "TOKEN_FILE": os.path.join(os.getcwd(), "token.json"),
    "SHEET_ID": "1z9GTxf9bLr9iGttZlYIT41qgRBfEyyFy10j788rs44Q",  # ← paste your Google Sheet ID here
    "SHEET_TAB_NAME": "Cases",
    "ARCHIVE_TAB_NAME": "Archive",
    "TIMEZONE": "Asia/Ho_Chi_Minh",
}

# Set your Slack webhook URL here, or set the env var SLACK_WEBHOOK_URL before launching Jupyter
SLACK_WEBHOOK_URL = ("https://hooks.slack.com/triggers/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", "")

SCOPES = [
    "https://www.googleapis.com/auth/gmail.modify",
    "https://www.googleapis.com/auth/spreadsheets",
]
TZ = ZoneInfo(CONFIG["TIMEZONE"])

print("Config loaded")
print(f"  DB:          {CONFIG['DB_FILE']}")
print(f"  Credentials: {CONFIG['CREDENTIALS_FILE']}")
print(f"  Token:       {CONFIG['TOKEN_FILE']}")
print(f"  Sheet ID:    {CONFIG['SHEET_ID']}")
print(f"  Slack set:   {'Yes' if SLACK_WEBHOOK_URL else 'No'}")

Config loaded
  DB:          c:\Users\vy.nguyenth\gmail-classifer\cases.db
  Credentials: c:\Users\vy.nguyenth\gmail-classifer\credentials.json
  Token:       c:\Users\vy.nguyenth\gmail-classifer\token.json
  Sheet ID:    1z9GTxf9bLr9iGttZlYIT41qgRBfEyyFy10j788rs44Q
  Slack set:   Yes


---
## 2. Queue Classification

In [54]:
QUEUES = [
    {
        "name": "Internal Invoice",
        "keywords": ["internal invoice"],
        "match_subject_only": True,
    },
    {
        "name": "Dispute",
        "keywords": ["chênh lệch", "sai sót", "bảng kê", "thiếu", "biên bản điều chỉnh", "dispute","Đối chiếu","sai lệch","cấn trừ"],
        "match_subject_only": False,
    },
    {
        "name": "Update Details",
        "keywords": ["thông tin", "không chính xác", "thay đổi", "update details", "update info"],
        "match_subject_only": False,
    },
    {
        "name": "Invoice",
        "keywords": ["Request invoice", "xuất hóa đơn", "chưa nhận được hóa đơn", "invoice request" ,"cung cấp hóa đơn","cung cấp"],
        "match_subject_only": False,
    },
]

def classify_email(subject: str, body: str) -> str:
    subject_lower = subject.lower()
    full_text = (subject + " " + body).lower()
    for queue in QUEUES:
        haystack = subject_lower if queue["match_subject_only"] else full_text
        for kw in queue["keywords"]:
            if kw.lower() in haystack:
                return queue["name"]
    return "Others"

print("classify_email defined")

classify_email defined


---
## 3. Database

In [55]:
def get_db() -> sqlite3.Connection:
    conn = sqlite3.connect(CONFIG["DB_FILE"])
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    with get_db() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS cases (
                case_id       TEXT PRIMARY KEY,
                message_id    TEXT UNIQUE NOT NULL,
                date_received TEXT,
                sender        TEXT,
                subject       TEXT,
                queue         TEXT,
                status        TEXT DEFAULT 'New',
                assigned_to   TEXT DEFAULT '',
                assigned_at   TEXT DEFAULT '',
                completed_at  TEXT DEFAULT '',
                email_link    TEXT,
                archived      INTEGER DEFAULT 0
            )
        """)
        conn.execute("CREATE INDEX IF NOT EXISTS idx_message_id ON cases(message_id)")
        conn.execute("CREATE INDEX IF NOT EXISTS idx_status ON cases(status)")

def get_existing_message_ids() -> set:
    with get_db() as conn:
        rows = conn.execute("SELECT message_id FROM cases").fetchall()
    return {r["message_id"] for r in rows}

def generate_case_id() -> str:
    now = datetime.now(TZ)
    return f"CASE-{now.strftime('%Y%m%d')}-{str(int(time.time() * 1000))[-5:]}"

def insert_cases(rows: list[dict]):
    with get_db() as conn:
        conn.executemany("""
            INSERT OR IGNORE INTO cases
              (case_id, message_id, date_received, sender, subject, queue,
               status, assigned_to, assigned_at, completed_at, email_link)
            VALUES
              (:case_id, :message_id, :date_received, :sender, :subject, :queue,
               'New', '', '', '', :email_link)
        """, rows)

def find_case(case_id: str):
    with get_db() as conn:
        return conn.execute("SELECT * FROM cases WHERE case_id = ?", (case_id,)).fetchone()

def update_case(case_id: str, **fields):
    if not fields:
        return
    set_clause = ", ".join(f"{k} = ?" for k in fields)
    values = list(fields.values()) + [case_id]
    with get_db() as conn:
        conn.execute(f"UPDATE cases SET {set_clause} WHERE case_id = ?", values)

init_db()
print(f"Database ready: {CONFIG['DB_FILE']}")

Database ready: c:\Users\vy.nguyenth\gmail-classifer\cases.db


---
## 4. Gmail Auth & Helpers

**How auth works:**
- First run → browser opens once for you to approve Gmail + Sheets access → `token.json` saved
- All subsequent runs → silent headless auto-refresh, no browser
- If `token.json` is deleted → browser opens again once

In [56]:
def _get_or_refresh_creds() -> Credentials:
    """Load token.json; refresh silently if expired; run browser flow if missing."""
    creds = None
    token_file = CONFIG["TOKEN_FILE"]
    if os.path.exists(token_file):
        creds = Credentials.from_authorized_user_file(token_file, SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                CONFIG["CREDENTIALS_FILE"], SCOPES
            )
            creds = flow.run_local_server(port=0)
        with open(token_file, "w") as f:
            f.write(creds.to_json())
    return creds

def get_gmail_service():
    return build("gmail", "v1", credentials=_get_or_refresh_creds())

def get_sheets_service():
    return build("sheets", "v4", credentials=_get_or_refresh_creds())

def get_or_create_label(service, name: str) -> str:
    labels = service.users().labels().list(userId="me").execute().get("labels", [])
    for label in labels:
        if label["name"] == name:
            return label["id"]
    created = service.users().labels().create(
        userId="me",
        body={"name": name, "labelListVisibility": "labelShow", "messageListVisibility": "show"},
    ).execute()
    return created["id"]

def apply_label(service, message_id: str, label_id: str):
    service.users().messages().modify(
        userId="me",
        id=message_id,
        body={"addLabelIds": [label_id]},
    ).execute()

def get_message_body(payload: dict) -> str:
    if payload.get("mimeType") == "text/plain":
        data = payload.get("body", {}).get("data", "")
        return base64.urlsafe_b64decode(data + "==").decode("utf-8", errors="replace") if data else ""
    for part in payload.get("parts", []):
        text = get_message_body(part)
        if text:
            return text
    return ""

print("Auth helpers defined")
print("First call to get_gmail_service() or get_sheets_service() will open browser if token.json is missing/expired")

Auth helpers defined
First call to get_gmail_service() or get_sheets_service() will open browser if token.json is missing/expired


---
## 5. Core Polling Logic

In [57]:
def check_new_emails():
    log.info("=== Polling Gmail ===")
    start = time.time()

    try:
        service = get_gmail_service()
    except Exception as e:
        log.error(f"Gmail auth failed: {e}")
        return

    label_id = get_or_create_label(service, CONFIG["PROCESSED_LABEL"])
    existing_ids = get_existing_message_ids()
    log.info(f"Loaded {len(existing_ids)} existing message IDs from DB.")

    query = f"to:{CONFIG['GROUP_EMAIL']} -label:{CONFIG['PROCESSED_LABEL']}"
    try:
        result = service.users().messages().list(
            userId="me", q=query, maxResults=CONFIG["MAX_RESULTS"]
        ).execute()
    except Exception as e:
        log.error(f"Gmail search failed: {e}")
        return

    messages = result.get("messages", [])
    log.info(f"Found {len(messages)} unprocessed messages.")

    new_rows, slack_queue, labeled_ids = [], [], []
    skipped = 0

    for msg_ref in messages:
        if time.time() - start > CONFIG["TIMEOUT_BUFFER_SECONDS"]:
            log.warning("Timeout buffer reached. Stopping early.")
            break

        msg_id = msg_ref["id"]
        if msg_id in existing_ids:
            skipped += 1
            continue

        try:
            msg = service.users().messages().get(
                userId="me", id=msg_id, format="full"
            ).execute()
        except Exception as e:
            log.warning(f"Could not fetch message {msg_id}: {e}")
            continue

        headers = {h["name"].lower(): h["value"] for h in msg.get("payload", {}).get("headers", [])}
        subject = headers.get("subject", "(no subject)")
        sender  = headers.get("from", "")
        date_ms = int(msg.get("internalDate", 0))
        date    = datetime.fromtimestamp(date_ms / 1000, tz=TZ)
        body    = get_message_body(msg.get("payload", {}))
        queue   = classify_email(subject, body)
        case_id = generate_case_id()
        email_link = f"https://mail.google.com/mail/u/0/#inbox/{msg_id}"

        row = {
            "case_id": case_id,
            "message_id": msg_id,
            "date_received": date.strftime("%d/%m/%Y %H:%M"),
            "sender": sender,
            "subject": subject,
            "queue": queue,
            "email_link": email_link,
        }
        new_rows.append(row)
        slack_queue.append(row)
        existing_ids.add(msg_id)
        labeled_ids.append(msg_id)

    if new_rows:
        insert_cases(new_rows)
        log.info(f"Wrote {len(new_rows)} new cases to DB.")

    for mid in labeled_ids:
        try:
            apply_label(service, mid, label_id)
        except Exception as e:
            log.warning(f"Could not label {mid}: {e}")

    send_slack_batch(slack_queue)
    sync_to_sheet()

    duration = time.time() - start
    log.info(f"=== Done in {duration:.1f}s | New: {len(new_rows)} | Skipped: {skipped} ===")

print("check_new_emails defined")

check_new_emails defined


---
## 6. Archive

In [59]:
def _parse_completed_at(raw: str):
    if not raw:
        return None
    try:
        return datetime.strptime(raw, "%d/%m/%Y %H:%M").replace(tzinfo=TZ)
    except ValueError:
        pass
    try:
        return datetime.fromisoformat(raw).replace(tzinfo=TZ)
    except ValueError:
        return None

def archive_old_completed():
    log.info("=== Archive run ===")
    cutoff = datetime.now(TZ) - timedelta(days=CONFIG["ARCHIVE_AFTER_DAYS"])

    with get_db() as conn:
        rows = conn.execute(
            "SELECT * FROM cases WHERE status = 'Completed' AND archived = 0"
        ).fetchall()

    archived = 0
    for row in rows:
        completed_dt = _parse_completed_at(row["completed_at"])
        if completed_dt is None:
            log.warning(f"Could not parse completed_at for {row['case_id']}: {row['completed_at']!r}")
            continue
        if completed_dt < cutoff:
            update_case(row["case_id"], archived=1)
            archived += 1

    log.info(f"Archived {archived} cases older than {CONFIG['ARCHIVE_AFTER_DAYS']} day(s).")

print("archive_old_completed defined")

archive_old_completed defined


---
## 7. Slack Notifications

In [60]:
QUEUE_EMOJI = {
    "Dispute": "🚨",
    "Update Details": "📝",
    "Invoice": "🧾",
    "Internal Invoice": "🏢",
    "Others": "📨",
}

def send_slack_raw(text: str):
    if not SLACK_WEBHOOK_URL:
        log.warning("SLACK_WEBHOOK_URL not set — skipping Slack notification.")
        return
    try:
        resp = requests.post(SLACK_WEBHOOK_URL, json={"text": text}, timeout=10)
        if resp.status_code != 200:
            log.warning(f"Slack webhook returned {resp.status_code}: {resp.text}")
    except Exception as e:
        log.error(f"Slack webhook error: {e}")

def send_slack_notification(case: dict):
    emoji = QUEUE_EMOJI.get(case["queue"], "📨")
    text = (
        f"{emoji} *New {case['queue']} Case*\n"
        f"*Case ID:* {case['case_id']}\n"
        f"*From:* {case['sender']}\n"
        f"*Subject:* {case['subject']}\n"
        f"*Email:* {case['email_link']}"
    )
    send_slack_raw(text)

def send_slack_batch(items: list[dict]):
    if not items:
        return
    if len(items) <= 5:
        for item in items:
            send_slack_notification(item)
            time.sleep(0.2)
        return
    by_queue: dict[str, int] = {}
    for item in items:
        by_queue[item["queue"]] = by_queue.get(item["queue"], 0) + 1
    lines = [f"📥 *{len(items)} new cases received*", "━" * 24]
    for q, count in by_queue.items():
        lines.append(f"{QUEUE_EMOJI.get(q, '📨')} *{q}:* {count}")
    lines += ["", "First few cases:"]
    for item in items[:5]:
        lines.append(f"• `{item['case_id']}` - {item['subject']}")
    if len(items) > 5:
        lines.append(f"...and {len(items) - 5} more.")
    send_slack_raw("\n".join(lines))

print("Slack helpers defined")

Slack helpers defined


---
## 7.5. Google Sheets Sync

Automatically called after every poll. Also run the cell below to push manually.

In [ ]:
SHEET_COLUMNS = [
    "Case ID", "Date Received", "Sender", "Subject", "Queue",
    "Status", "Assigned To", "Assigned At", "Completed At", "Email Link",
]

def _ensure_tab_exists(service, spreadsheet_id: str, tab_name: str):
    meta = service.spreadsheets().get(spreadsheetId=spreadsheet_id).execute()
    existing = [s["properties"]["title"] for s in meta.get("sheets", [])]
    if tab_name not in existing:
        service.spreadsheets().batchUpdate(
            spreadsheetId=spreadsheet_id,
            body={"requests": [{"addSheet": {"properties": {"title": tab_name}}}]},
        ).execute()
        print(f"Created sheet tab: {tab_name}")

def _write_tab(service, sheet_id: str, tab_name: str, rows):
    _ensure_tab_exists(service, sheet_id, tab_name)
    service.spreadsheets().values().clear(
        spreadsheetId=sheet_id, range=f"{tab_name}!A:Z"
    ).execute()
    service.spreadsheets().values().update(
        spreadsheetId=sheet_id,
        range=f"{tab_name}!A1",
        valueInputOption="RAW",
        body={"values": rows},
    ).execute()

def sync_to_sheet():
    sheet_id = CONFIG["SHEET_ID"]
    if sheet_id == "YOUR_SHEET_ID_HERE":
        log.warning("SHEET_ID not configured — set it in CONFIG above and re-run.")
        return

    try:
        service = get_sheets_service()
        # Active cases → "Cases" tab
        with get_db() as conn:
            active_rows = conn.execute(
                "SELECT case_id, date_received, sender, subject, queue, "
                "status, assigned_to, assigned_at, completed_at, email_link "
                "FROM cases WHERE archived = 0 ORDER BY substr(date_received, 7, 4) || substr(date_received, 4, 2) || substr(date_received, 1, 2) DESC"
            ).fetchall()
        active_values = [SHEET_COLUMNS] + [
            [r["case_id"], r["date_received"], r["sender"], r["subject"],
             r["queue"], r["status"], r["assigned_to"], r["assigned_at"],
             r["completed_at"], r["email_link"]] for r in active_rows
        ]
        _write_tab(service, sheet_id, CONFIG["SHEET_TAB_NAME"], active_values)
        log.info(f"Synced {len(active_rows)} active cases → '{CONFIG['SHEET_TAB_NAME']}' tab.")
        # Archived cases → "Archive" tab
        with get_db() as conn:
            archive_rows = conn.execute(
                "SELECT case_id, date_received, sender, subject, queue, "
                "status, assigned_to, assigned_at, completed_at, email_link "
                "FROM cases WHERE archived = 1 ORDER BY completed_at DESC"
            ).fetchall()
        archive_values = [SHEET_COLUMNS] + [
            [r["case_id"], r["date_received"], r["sender"], r["subject"],
             r["queue"], r["status"], r["assigned_to"], r["assigned_at"],
             r["completed_at"], r["email_link"]] for r in archive_rows
        ]
        _write_tab(service, sheet_id, CONFIG["ARCHIVE_TAB_NAME"], archive_values)
        log.info(f"Synced {len(archive_rows)} archived cases → '{CONFIG['ARCHIVE_TAB_NAME']}' tab.")
    except Exception as e:
        log.error(f"Sheet sync failed: {e}")

print("sync_to_sheet defined")

sync_to_sheet defined


In [67]:
# Run this cell to manually push current DB state to Google Sheets
sync_to_sheet()

2026-05-24 11:15:57  ERROR   Sheet sync failed: cannot access local variable 'get_sheets_service' where it is not associated with a value


---
## 8. Actions — Assign / Complete / Reclassify

In [11]:
def assign_case(case_id: str, user: str | None = None):
    user = user or os.environ.get("USER", "unknown")
    now  = datetime.now(TZ).strftime("%d/%m/%Y %H:%M")
    case = find_case(case_id)
    if not case:
        print(f"Case not found: {case_id}")
        return
    update_case(case_id, status="In Progress", assigned_to=user, assigned_at=now)
    print(f"Assigned {case_id} to {user}.")

def complete_case(case_id: str):
    now  = datetime.now(TZ).strftime("%d/%m/%Y %H:%M")
    case = find_case(case_id)
    if not case:
        print(f"Case not found: {case_id}")
        return
    update_case(case_id, status="Completed", completed_at=now)
    print(f"Completed {case_id}.")

def reclassify_case(case_id: str, queue: str):
    case = find_case(case_id)
    if not case:
        print(f"Case not found: {case_id}")
        return
    update_case(case_id, queue=queue)
    print(f"Reclassified {case_id} → {queue}.")

print("Action helpers defined")

Action helpers defined


---
## 9. Run: Poll Once
Checks Gmail right now, writes new cases to DB, then syncs to Google Sheet.

In [37]:
check_new_emails()

2026-05-24 09:49:03  INFO    === Polling Gmail ===
2026-05-24 09:49:03  INFO    file_cache is only supported with oauth2client<4.0.0
2026-05-24 09:49:04  INFO    Loaded 0 existing message IDs from DB.
2026-05-24 09:49:04  INFO    Found 0 unprocessed messages.
2026-05-24 09:49:04  INFO    file_cache is only supported with oauth2client<4.0.0
2026-05-24 09:49:06  INFO    Synced 0 active cases → 'Cases' tab.
2026-05-24 09:49:07  INFO    Synced 0 archived cases → 'Archive' tab.
2026-05-24 09:49:07  INFO    === Done in 4.3s | New: 0 | Skipped: 0 ===


---
## 10. Run: List Cases
Change `status_filter` to `"New"`, `"In Progress"`, `"Completed"`, or `None` for all.

In [41]:
# remove current existing label

def remove_dispute_labels_from_inbox():
    service = get_gmail_service()

    labels = service.users().labels().list(userId="me").execute().get("labels", [])
    label_map = {label["name"]: label["id"] for label in labels}

    target_names = ["dispute-tracker", "dispute-tracker-processed"]
    remove_label_ids = [label_map[name] for name in target_names if name in label_map]

    if not remove_label_ids:
        print("No matching labels found:", target_names)
        return

    query = "in:inbox (label:dispute-tracker OR label:dispute-tracker-processed)"
    total = 0
    next_page_token = None

    while True:
        response = service.users().messages().list(
            userId="me",
            q=query,
            pageToken=next_page_token,
            maxResults=500,
        ).execute()

        messages = response.get("messages", [])
        if not messages:
            break

        for msg in messages:
            service.users().messages().modify(
                userId="me",
                id=msg["id"],
                body={"removeLabelIds": remove_label_ids},
            ).execute()
            total += 1

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

    print(f"Removed dispute labels from {total} inbox messages.")

    remove_dispute_labels_from_inbox()

In [43]:
def sync_from_date(date_str="2026/05/16"):
    service = get_gmail_service()
    query = f"in:anywhere to:{CONFIG['GROUP_EMAIL']} after:{date_str}"
    result = service.users().messages().list(
        userId="me",
        q=query,
        maxResults=500,
    ).execute()
    messages = result.get("messages", [])
    print("Found", len(messages), "messages since", date_str)
    return messages

In [44]:
msgs = sync_from_date("2026/05/16")

2026-05-24 09:58:22  INFO    file_cache is only supported with oauth2client<4.0.0


Found 64 messages since 2026/05/16


In [ ]:
#rerun sync with classification and DB insertion from 16/05/2026

def check_new_emails_from_date(date_str="2026/05/16"):
    log.info("=== Polling Gmail from date ===")
    service = get_gmail_service()
    label_id = get_or_create_label(service, CONFIG["PROCESSED_LABEL"])
    existing_ids = get_existing_message_ids()

    query = f"in:anywhere to:{CONFIG['GROUP_EMAIL']} after:{date_str}"
    result = service.users().messages().list(
        userId="me", q=query, maxResults=500
    ).execute()

    messages = result.get("messages", [])
    log.info(f"Found {len(messages)} messages since {date_str}.")

    new_rows, slack_queue, labeled_ids = [], [], []
    for msg_ref in messages:
        msg_id = msg_ref["id"]
        if msg_id in existing_ids:
            continue

        msg = service.users().messages().get(
            userId="me", id=msg_id, format="full"
        ).execute()

        headers = {h["name"].lower(): h["value"] for h in msg.get("payload", {}).get("headers", [])}
        subject = headers.get("subject", "(no subject)")
        sender = headers.get("from", "")
        date_ms = int(msg.get("internalDate", 0))
        date = datetime.fromtimestamp(date_ms / 1000, tz=TZ)
        body = get_message_body(msg.get("payload", {}))
        queue = classify_email(subject, body)
        case_id = generate_case_id()
        email_link = f"https://mail.google.com/mail/u/0/#inbox/{msg_id}"

        row = {
            "case_id": case_id,
            "message_id": msg_id,
            "date_received": date.strftime("%d/%m/%Y %H:%M"),
            "sender": sender,
            "subject": subject,
            "queue": queue,
            "email_link": email_link,
        }
        new_rows.append(row)
        slack_queue.append(row)
        existing_ids.add(msg_id)
        labeled_ids.append(msg_id)

    if new_rows:
        insert_cases(new_rows)
        log.info(f"Wrote {len(new_rows)} new cases to DB.")

    for mid in labeled_ids:
        try:
            apply_label(service, mid, label_id)
        except Exception as e:
            log.warning(f"Could not label {mid}: {e}")

    send_slack_batch(slack_queue)
    sync_to_sheet()

    log.info(f"=== Done | New: {len(new_rows)} ===")

In [48]:
check_new_emails_from_date("2026/05/16")

2026-05-24 11:10:00  INFO    === Polling Gmail from date ===
2026-05-24 11:10:00  INFO    file_cache is only supported with oauth2client<4.0.0
2026-05-24 11:10:01  INFO    Found 64 messages since 2026/05/16.
2026-05-24 11:10:01  ERROR   Sheet sync failed: cannot access local variable 'get_sheets_service' where it is not associated with a value
2026-05-24 11:10:01  INFO    === Done | New: 0 ===


In [ ]:
status_filter = None  # "New" | "In Progress" | "Completed" | None

with get_db() as conn:
    if status_filter:
        rows = conn.execute(
            "SELECT * FROM cases WHERE archived = 0 AND status = ? ORDER BY substr(date_received, 7, 4) || substr(date_received, 4, 2) || substr(date_received, 1, 2) DESC",
            (status_filter,),
        ).fetchall()
    else:
        rows = conn.execute(
            "SELECT * FROM cases WHERE archived = 0 ORDER BY substr(date_received, 7, 4) || substr(date_received, 4, 2) || substr(date_received, 1, 2) DESC"
        ).fetchall()

if not rows:
    print("No cases found.")
else:
    print(f"{'Case ID':<22} {'Date':<17} {'Queue':<18} {'Status':<12} Sender")
    print("-" * 100)
    for r in rows:
        print(f"{r['case_id']:<22} {r['date_received']:<17} {r['queue']:<18} {r['status']:<12} {r['sender'][:40]}")
    print(f"\n{len(rows)} case(s).")

No cases found.


### Assign / Complete / Reclassify a case
Fill in the `case_id` and run the relevant cell.

In [39]:
assign_case("CASE-20261003-12345")
# assign_case("CASE-XXXXXXXX-XXXXX", "colleague@grabtaxi.com")

Case not found: CASE-20261003-12345


In [ ]:
complete_case("CASE-XXXXXXXX-XXXXX")

In [ ]:
# Queues: Dispute | Invoice | Update Details | Internal Invoice | Others
reclassify_case("CASE-XXXXXXXX-XXXXX", "Invoice")

---
## 11. Diagnostics

In [ ]:
# ── Health check ─────────────────────────────────────────────────────────────
print("=== Dispute Tracker Diagnostic ===")

print("✅ SLACK_WEBHOOK_URL is set" if SLACK_WEBHOOK_URL else "⚠️  SLACK_WEBHOOK_URL not set")

try:
    with get_db() as conn:
        total    = conn.execute("SELECT COUNT(*) FROM cases WHERE archived = 0").fetchone()[0]
        archived = conn.execute("SELECT COUNT(*) FROM cases WHERE archived = 1").fetchone()[0]
    print(f"✅ Database OK — {total} active cases, {archived} archived")
except Exception as e:
    print(f"❌ DB error: {e}")

try:
    service = get_gmail_service()
    profile = service.users().getProfile(userId="me").execute()
    print(f"✅ Gmail authenticated as {profile['emailAddress']}")
    query  = f"to:{CONFIG['GROUP_EMAIL']} -label:{CONFIG['PROCESSED_LABEL']}"
    result = service.users().messages().list(userId="me", q=query, maxResults=5).execute()
    print(f"📧 Unprocessed emails (estimate): {result.get('resultSizeEstimate', 0)}")
except Exception as e:
    print(f"❌ Gmail error: {e}")

print(f"\nSheet ID configured: {'No — set SHEET_ID in CONFIG' if CONFIG['SHEET_ID'] == 'YOUR_SHEET_ID_HERE' else CONFIG['SHEET_ID']}")
print("=== Done ===")

In [ ]:
# ── Debug archive (dry run) ───────────────────────────────────────────────────
print("=== Archive Debug ===")
cutoff = datetime.now(TZ) - timedelta(days=CONFIG["ARCHIVE_AFTER_DAYS"])
print(f"Cutoff: cases completed before {cutoff.strftime('%d/%m/%Y')} will be archived")
print(f"Today:  {datetime.now(TZ).strftime('%d/%m/%Y')}\n")

with get_db() as conn:
    rows = conn.execute(
        "SELECT * FROM cases WHERE status = 'Completed' AND archived = 0"
    ).fetchall()

if not rows:
    print("No completed cases.")
else:
    would_archive = 0
    for row in rows:
        completed_dt = _parse_completed_at(row["completed_at"])
        if completed_dt is None:
            print(f"⚠️  {row['case_id']} — could not parse: {row['completed_at']!r}")
            continue
        age_days = (datetime.now(TZ) - completed_dt).days
        if completed_dt < cutoff:
            would_archive += 1
            print(f"✅ {row['case_id']} — {row['completed_at']} ({age_days}d ago) → WOULD ARCHIVE")
        else:
            print(f"⏳ {row['case_id']} — {row['completed_at']} ({age_days}d ago) → keep")
    print(f"\nTotal completed: {len(rows)} | Would archive: {would_archive}")

print("=== Done ===")

In [ ]:
# ── DB row health check ───────────────────────────────────────────────────────
print("=== DB Row Health Check ===")

with get_db() as conn:
    rows = conn.execute("SELECT * FROM cases WHERE archived = 0").fetchall()

empty_rows = in_progress_rows = in_progress_empty = 0
for row in rows:
    if not row["case_id"] or not row["case_id"].strip():
        empty_rows += 1
        print(f"⚠️  Empty case_id: message_id={row['message_id']!r}")
    if row["status"] == "In Progress":
        in_progress_rows += 1
        if not row["case_id"]:
            in_progress_empty += 1
            print(f"❌ In Progress row has no case_id! message_id={row['message_id']!r}")

print(f"\nTotal active rows:           {len(rows)}")
print(f"Empty rows (no case_id):      {empty_rows}")
print(f"Total In Progress:            {in_progress_rows}")
print(f"In Progress with no case_id:  {in_progress_empty}")
print("=== Done ===")

---
## 12. Tests

In [ ]:
# ── Test classification ───────────────────────────────────────────────────────
TEST_CASES = [
    ("Khiếu nại chênh lệch hóa đơn tháng 4", "chênh lệch trong bảng kê",     "Dispute"),
    ("Yêu cầu cập nhật thông tin công ty",    "thông tin không chính xác",     "Update Details"),
    ("Request Invoice for March 2025",         "chưa nhận được hóa đơn",       "Invoice"),
    ("Internal Invoice Q1 Settlement",         "internal invoice between ents", "Internal Invoice"),
    ("General inquiry about payment",          "question about my payment",     "Others"),
]

passed = 0
for subject, body, expected in TEST_CASES:
    got = classify_email(subject, body)
    ok  = got == expected
    passed += ok
    note = "" if ok else f" (expected {expected})"
    print(f"[{'PASS' if ok else 'FAIL'}] \"{subject[:45]}\" → {got}{note}")

print(f"\n{passed}/{len(TEST_CASES)} passed.")

In [ ]:
# ── Test archive (end-to-end) ─────────────────────────────────────────────────
now = datetime.now(TZ)
old_completed_at   = (now - timedelta(days=5)).strftime("%d/%m/%Y %H:%M")
today_completed_at = now.strftime("%d/%m/%Y %H:%M")
today_received     = now.strftime("%d/%m/%Y")

test_rows = [
    {"case_id": "ARCHTEST-001", "message_id": "archtest-msg-001", "date_received": today_received,
     "sender": "old.1@test.com", "subject": "[ARCHTEST] Old case 1", "queue": "Dispute", "email_link": "https://mail.google.com"},
    {"case_id": "ARCHTEST-002", "message_id": "archtest-msg-002", "date_received": today_received,
     "sender": "old.2@test.com", "subject": "[ARCHTEST] Old case 2", "queue": "Invoice", "email_link": "https://mail.google.com"},
    {"case_id": "ARCHTEST-003", "message_id": "archtest-msg-003", "date_received": today_received,
     "sender": "today@test.com", "subject": "[ARCHTEST] Today completed", "queue": "Invoice", "email_link": "https://mail.google.com"},
    {"case_id": "ARCHTEST-004", "message_id": "archtest-msg-004", "date_received": today_received,
     "sender": "stuck@test.com", "subject": "[ARCHTEST] In progress", "queue": "Others", "email_link": "https://mail.google.com"},
    {"case_id": "ARCHTEST-005", "message_id": "archtest-msg-005", "date_received": today_received,
     "sender": "new@test.com", "subject": "[ARCHTEST] Brand new", "queue": "Others", "email_link": "https://mail.google.com"},
]

insert_cases(test_rows)
update_case("ARCHTEST-001", status="Completed", completed_at=old_completed_at)
update_case("ARCHTEST-002", status="Completed", completed_at=old_completed_at)
update_case("ARCHTEST-003", status="Completed", completed_at=today_completed_at)
update_case("ARCHTEST-004", status="In Progress")

with get_db() as conn:
    before_active   = conn.execute("SELECT COUNT(*) FROM cases WHERE archived = 0").fetchone()[0]
    before_archived = conn.execute("SELECT COUNT(*) FROM cases WHERE archived = 1").fetchone()[0]

print(f"BEFORE: active={before_active}, archived={before_archived}")
archive_old_completed()

with get_db() as conn:
    after_active   = conn.execute("SELECT COUNT(*) FROM cases WHERE archived = 0").fetchone()[0]
    after_archived = conn.execute("SELECT COUNT(*) FROM cases WHERE archived = 1").fetchone()[0]

print(f"AFTER:  active={after_active}, archived={after_archived}")
active_delta = after_active - before_active
arch_delta   = after_archived - before_archived

if active_delta == 3 and arch_delta == 2:
    print(f"✅ PASS: active +{active_delta} (expected +3), archived +{arch_delta} (expected +2)")
else:
    print(f"❌ FAIL: active delta={active_delta} (expected +3), archived delta={arch_delta} (expected +2)")

print("\nRun the cleanup cell below to remove test rows.")

In [ ]:
# ── Cleanup test data ─────────────────────────────────────────────────────────
with get_db() as conn:
    result = conn.execute(
        "DELETE FROM cases WHERE case_id LIKE 'TEST-%' OR case_id LIKE 'ARCHTEST-%'"
    )
print(f"Cleaned up {result.rowcount} test row(s).")

In [ ]:
pip install -r requirements.txt
